# SHORE — Interactive Visualisation

This notebook generates a unit-sphere 6-block caps mesh from scratch, then
inspects the result using the OOP API and PyVista's HTML backend.

**Prerequisites**
```bash
pip install shore[vis]   # adds pyvista
```

In [ ]:
import numpy as np
import pyvista as pv

from shore.mesh import Body, CapsTopology, Mesh, Spacing1D
from shore.mesh import info_geo, check_geo, export_geo, export_surface
from shore.io.geo import read_geo
from shore.io.vtk import open_to_pyvista

pv.set_jupyter_backend('html')

## 0 — Generate a unit-sphere 6-block caps mesh

Creates `sphere.stl`, projects onto a structured `(ni×nj)` surface layer
saved as `sphere_k0.npz`, then marches `nk` wall-normal layers to produce
6 `.geo` files. Skips any step whose output already exists.

In [ ]:
from pathlib import Path
import trimesh

NI, NJ, NK = 40, 60, 20
DS, GROWTH = 0.02, 1.10

STL_PATH = Path('sphere.stl')
NPZ_PATH = Path('sphere_k0.npz')
GEO_STEM = Path('sphere')

LABELS   = ['sub0', 'sub1', 'sub2', 'sub3', 'cap_north', 'cap_south']
GEO_PATHS = [Path(f'sphere_{lbl}.geo') for lbl in LABELS]

# --- STL ---
if not STL_PATH.exists():
    m = trimesh.creation.icosphere(subdivisions=4, radius=1.0)
    m.export(str(STL_PATH))
    print(f'Generated {STL_PATH}  ({len(m.faces)} triangles)')
else:
    print(f'{STL_PATH} already exists, skipping.')

body = Body.from_stl(STL_PATH)

# --- surface projection ---
if not NPZ_PATH.exists():
    k0 = body.project(NI, NJ)
    Body.save_surface(k0, NPZ_PATH)
    print(f'Generated {NPZ_PATH}  shape={k0.shape}')
else:
    print(f'{NPZ_PATH} already exists, skipping.')
    k0 = Body.load_surface(NPZ_PATH)

# --- march ---
if not GEO_PATHS[0].exists():
    topo   = CapsTopology()
    blocks = topo.build(k0, NK)
    m      = Mesh(body=body, blocks=blocks, topology=topo)

    def _cb(k: int, jmin: float) -> None:
        print(f'  layer {k:3d}  jmin={jmin:.3e}', end='\r')

    m.march(ds=DS, growth=GROWTH, layer_callback=_cb)
    print()
    m.write_geo(GEO_STEM)
    for blk in m.blocks:
        print(f'  sphere_{blk.label}.geo  shape={blk.nodes.shape[:3]}')
else:
    print('Geo files already exist, skipping marching.')

## 1 — Inspect block metadata

In [ ]:
from shore.mesh import info_geo

for p in GEO_PATHS:
    d  = info_geo(p)
    ni, nj, nk = d['shape']
    print(f"{p.name:25s}  ni={ni}  nj={nj}  nk={nk}  "
          f"points={d['n_points']:,}  cells={d['n_cells']:,}")

## 2 — Jacobian check per block

In [ ]:
from shore.mesh import check_geo

for p in GEO_PATHS:
    result = check_geo(p)
    jmins  = result['jmins']
    status = 'OK' if result['ok'] else 'INVERTED'
    print(f"{p.name:25s}  global_jmin={min(jmins):.4e}  status={status}")

## 3 — Per-layer Jacobian profile

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3))

for p, lbl in zip(GEO_PATHS, LABELS):
    jmins = check_geo(p)['jmins']
    ax.plot(range(1, len(jmins) + 1), jmins, marker='o', ms=3, lw=1.2, label=lbl)

ax.axhline(0, color='red', lw=0.8, ls='--', label='inversion threshold')
ax.set_xlabel('layer k')
ax.set_ylabel('jmin')
ax.set_title('Minimum cell Jacobian per layer — all 6 blocks')
ax.legend(fontsize=8, ncol=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4 — Visualise one sub-block coloured by Jacobian

In [ ]:
from shore.mesh import view_geo_file

pl = view_geo_file(GEO_PATHS[0], colorby='Jacobian', jupyter=True)
pl.show()

## 5 — Visualise all 6 blocks as a MultiBlock in PyVista

In [ ]:
from shore.io.geo import read_geo
from shore.io.vtk import open_to_pyvista

pv.set_jupyter_backend('html')
pl = pv.Plotter()

for p in GEO_PATHS:
    grid = read_geo(p)
    sg   = open_to_pyvista(grid, with_jacobian=True)
    pl.add_mesh(sg, scalars='Jacobian', cmap='RdYlGn',
                show_edges=True, opacity=0.7)

pl.add_axes()
pl.add_bounding_box()
pl.show()

## 6 — Surface layer (k=0)

In [ ]:
from shore.mesh import view_surface_file

# Display surface metadata
from shore.mesh import info_surface
d = info_surface(NPZ_PATH)
print(f"Surface shape : {d['shape']}")
print(f"Points        : {d['n_points']:,}")

pl = view_surface_file(NPZ_PATH, colorby='radius', jupyter=True)
pl.show()

## 7 — Export to VTK for ParaView

In [ ]:
from shore.mesh import export_geo, export_surface
from shore.io.vtk import write_caps_vtm

# Export surface
export_surface(NPZ_PATH)
print(f'Wrote sphere_k0.vts')

# Export all 6 blocks as a single MultiBlock
blocks_data = [read_geo(p) for p in GEO_PATHS]
write_caps_vtm('sphere.vtm', blocks_data, with_jacobian=True)
print('Wrote sphere.vtm — open in ParaView.')

## 8 — Spacing: surface clustering and wall-normal distributions

In [ ]:
from shore.mesh import Spacing1D

# Geometric (default)
sp_geom  = Spacing1D(law='geometric', ds=DS, growth=GROWTH)
k_geom   = sp_geom.build_steps(NK)

# Tanh: cluster near wall within fixed total thickness
sp_tanh  = Spacing1D(law='tanh', beta=4.0, total_thickness=2.0)
k_tanh   = sp_tanh.build_steps(NK)

# Tanh2: cluster at both wall and far-field
sp_tanh2 = Spacing1D(law='tanh2', beta=3.0, total_thickness=2.0)
k_tanh2  = sp_tanh2.build_steps(NK)

print(f'Geometric — first: {k_geom[0]:.4f}, last: {k_geom[-1]:.4f}, total: {k_geom.sum():.3f}')
print(f'Tanh      — first: {k_tanh[0]:.4f}, last: {k_tanh[-1]:.4f}, total: {k_tanh.sum():.3f}')
print(f'Tanh2     — first: {k_tanh2[0]:.4f}, last: {k_tanh2[-1]:.4f}, total: {k_tanh2.sum():.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
layers  = np.arange(1, NK)
ax.semilogy(layers, k_geom,  'o-', ms=3, label='geometric')
ax.semilogy(layers, k_tanh,  's-', ms=3, label='tanh (wall cluster)')
ax.semilogy(layers, k_tanh2, '^-', ms=3, label='tanh2 (wall + far-field)')
ax.set_xlabel('layer k')
ax.set_ylabel('step size')
ax.set_title('Wall-normal step-size distributions')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9 — Generate a mesh with tanh wall-normal spacing

In [ ]:
TANH_STEM  = Path('sphere_tanh')
TANH_SUB0  = Path('sphere_tanh_sub0.geo')

if not TANH_SUB0.exists():
    sp_k = Spacing1D(law='tanh', beta=4.0, total_thickness=2.0)
    topo   = CapsTopology()
    blocks = topo.build(k0, NK, spacing_k=sp_k)
    m_tanh = Mesh(body=body, blocks=blocks, topology=topo)
    m_tanh.march(ds=DS, growth=GROWTH, steps=sp_k.build_steps(NK))
    m_tanh.write_geo(TANH_STEM)
    print('Generated sphere_tanh_*.geo files.')
else:
    print('sphere_tanh_sub0.geo already exists, skipping.')

# Compare Jacobian profiles: geometric vs tanh
jmins_geom = check_geo(GEO_PATHS[0])['jmins']
jmins_tanh = check_geo(TANH_SUB0)['jmins']

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, len(jmins_geom) + 1), jmins_geom, 'o-', ms=4, lw=1.5, label='geometric (sub0)')
ax.plot(range(1, len(jmins_tanh) + 1), jmins_tanh, 's-', ms=4, lw=1.5, label='tanh beta=4 (sub0)')
ax.axhline(0, color='red', lw=0.8, ls='--')
ax.set_xlabel('layer k')
ax.set_ylabel('jmin')
ax.set_title('Jacobian profile: geometric vs tanh spacing (sub0 block)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()